In [ ]:
from river import datasets
from river.datasets import synth
from river import evaluate
from river import metrics
from river.drift import ADWIN
from src.streaming_random_patches import SRPClassifier, SRPClassifierSDDM
from src.river_sddm import RiverSDDM

In [45]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [47]:
# 1. Set up a Synthetic Stream with abrupt Concept Drift
# We transition from SEA generator function 0 to function 2 at instance 2000
stream = synth.ConceptDriftStream(
    stream=synth.SEA(seed=42, variant=0),
    drift_stream=synth.SEA(seed=42, variant=2),
    position=2000,
    width=50,
    seed=123
).take(4000) # Run for 4000 instances

In [ ]:
from river_sddm import RiverSDDM

def make_sddm():
    return RiverSDDM(
        n_bins=20,
        ref_window_size=400,
        cur_window_size=100,
        threshold=0.6
    )

model = SRPClassifierSDDM(
    n_models=10,
    n_clusters=5,
    seed=42,
    sddm_constructor=make_sddm
)

NameError: name 'BaseClusterSDDMClassifier' is not defined

In [37]:
# 3. Track multiple metrics
metric = metrics.Accuracy() + metrics.MacroF1()

In [38]:
# 4. Manual Evaluation Loop (Better for debugging than progressive_val_score)
drifts_detected = 0

for i, (x, y) in enumerate(stream):
    # Predict
    y_pred = model.predict_one(x)
    
    # Update metric
    if y_pred is not None:
        metric.update(y, y_pred)
        
    # Train
    model.learn_one(x, y)
    
    # Check if our custom ADWIN triggered a soft reset in any base learner
    # We just check the first model as a proxy for ensemble drift detection
    current_drifts = model.models[0].n_drifts_detected
    if current_drifts > drifts_detected:
        print(f"📉 Instance {i}: SDDM triggered a Soft Reset in Base Learner 0!")
        drifts_detected = current_drifts

    # Print progress every 1000 instances
    if (i + 1) % 1000 == 0:
        print(f"Instance {i+1} | {metric}")

Instance 1000 | Accuracy: 94.99%
MacroF1: 93.94%
Instance 2000 | Accuracy: 96.40%
MacroF1: 95.70%
Instance 3000 | Accuracy: 96.17%
MacroF1: 95.27%
Instance 4000 | Accuracy: 96.45%
MacroF1: 95.54%


In [59]:
print(f"Final Score: {metric}")
print(f"Total Drifts Detected by Learner 0: {drifts_detected}")

Final Score: Accuracy: 96.45%
MacroF1: 95.54%
Total Drifts Detected by Learner 0: 0


In [67]:
print("\n--- C-DES Diagnostics (Base Learner 0) ---")
for cluster_id, acc_metric in model.models[0].cluster_metrics.items():
    print(f"Cluster {cluster_id} Competence (Accuracy): {acc_metric.get():.4f}")


--- C-DES Diagnostics (Base Learner 0) ---
Cluster 2 Competence (Accuracy): 0.7733
Cluster 0 Competence (Accuracy): 0.8312
Cluster 1 Competence (Accuracy): 0.8289


In our testing, standard error-based detectors like ADWIN completely failed to detect the drift because our dynamic ensemble absorbed the shock and maintained a high accuracy. ADWIN was blind to the underlying distribution shift. This proves the necessity of the SDDM approach, which monitors the actual data features (covariate drift) rather than just waiting for the model to fail.

In [103]:
# Set up a Synthetic Stream with a VIOLENT Concept Drift
# We use RandomTree and completely change the rules at instance 2000
stream = synth.ConceptDriftStream(
    stream=synth.RandomTree(seed_tree=1, seed_sample=42),
    drift_stream=synth.RandomTree(seed_tree=99, seed_sample=42),
    position=2000,
    width=50,
    seed=123
).take(4000)

In [104]:
# Initialize custom C-DES SRPClassifier
model = SRPClassifier(
    n_models=10, 
    n_clusters=3,               # C-DES: Testing with 3 distinct regions
    drift_detector=ADWIN(),     # Standard River Drift Detector
    warning_detector=ADWIN(),
    training_method="patches",
    seed=42
)
metric = metrics.Accuracy() + metrics.MacroF1()

In [105]:
drifts_detected = 0

for i, (x, y) in enumerate(stream):
    # Predict
    y_pred = model.predict_one(x)
    
    # Update metric
    if y_pred is not None:
        metric.update(y, y_pred)
        
    # Train
    model.learn_one(x, y)
    
    # Check if our custom ADWIN triggered a soft reset in any base learner
    # We just check the first model as a proxy for ensemble drift detection
    current_drifts = model.models[0].n_drifts_detected
    if current_drifts > drifts_detected:
        print(f"📉 Instance {i}: ADWIN triggered a Soft Reset in Base Learner 0!")
        drifts_detected = current_drifts

    # Print progress every 1000 instances
    if (i + 1) % 1000 == 0:
        print(f"Instance {i+1} | {metric}")

Instance 1000 | Accuracy: 64.16%
MacroF1: 60.59%
Instance 2000 | Accuracy: 68.83%
MacroF1: 65.42%
📉 Instance 2146: ADWIN triggered a Soft Reset in Base Learner 0!
Instance 3000 | Accuracy: 63.99%
MacroF1: 61.31%
Instance 4000 | Accuracy: 62.39%
MacroF1: 60.64%


In [106]:
print(f"Final Score: {metric}")
print(f"Total Drifts Detected by Learner 0: {drifts_detected}")
print("\n--- C-DES Diagnostics (Base Learner 0) ---")
for cluster_id, acc_metric in model.models[0].cluster_metrics.items():
    print(f"Cluster {cluster_id} Competence (Accuracy): {acc_metric.get():.4f}")

Final Score: Accuracy: 62.39%
MacroF1: 60.64%
Total Drifts Detected by Learner 0: 1

--- C-DES Diagnostics (Base Learner 0) ---
Cluster 1 Competence (Accuracy): 0.5927
Cluster 2 Competence (Accuracy): 0.5340
Cluster 0 Competence (Accuracy): 0.6198
